<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/FinalWork/approach_3_seniority_jobbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Fine-tuned classification model

We build a prediction model based on approach 3 (Fine-tuned classification model) of the Assignment. We employ JobBERT-V3 in our approach, as it is specifically designed for feature extraction and downstream supervised learning.

We chose JobBERT as the underlying language model because current research shows that domain-specific representations of job titles substantially outperform generic sentence encoders for HR-related prediction tasks. Prior work demonstrates that JobBERT captures fine-grained semantic signals in job titles by leveraging large-scale co-occurrence information from vacancies and skills, making it particularly well suited for tasks such as job title normalization and organizational categorization (Decorte et al., 2021). Consequently, JobBERT provides an optimal foundation for predicting departments based solely on job titles.

The contrastive pretraining of JobBERT-V3 results in highly separable representations that are particularly well suited for fine-tuning classification models and for confidence-based decision thresholds in open-set inference settings.

JobBERT-V3 is a domain-specific, contrastively trained model for job titles that delivers robust semantic representations across multiple languages (English, German, Spanish, and Chinese) without requiring task-specific supervision. By combining large-scale multilingual training (21M+ job titles) with an efficiency-oriented architecture, it achieves state-of-the-art performance in both monolingual and cross-lingual job title matching

In [ ]:
# Load fine-tuned model
import os, json, joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = "artifacts/jobbert_dept_finetuned"

if os.path.isdir(SAVE_DIR) and os.path.isfile(f"{SAVE_DIR}/config.json"):
    print("✅ Loading fine-tuned model from local artifacts...")
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, fix_mistral_regex=True)
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

    le = joblib.load(f"{SAVE_DIR}/label_encoder.joblib")
    meta = json.load(open(f"{SAVE_DIR}/meta.json"))
else:
    print("⚠️ No saved model found. Training from scratch...")
    # ... fine-tuning code ...


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

import torch


In [ ]:
#Load Data
# Department dataset
SENIORITY_DATA_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/FinalWork/seniority.csv"
df_seniority = pd.read_csv(SENIORITY_DATA_PATH)

# Profiles dataset
PROFILES_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/df_profiles_cleansed.csv"
df_profiles = pd.read_csv(PROFILES_PATH)


# 1) We start with a baseline (logistic regression) to check if our then fine tuned model performs better than classical machine learning models.

#1.1 JobBERT embeddings + Logistic Regression

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/FinalWork/seniority.csv")

X = df["text"].astype(str).tolist()
y = df["label"].astype(str).tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

embedder = SentenceTransformer("TechWolf/JobBERT-v3")

X_train_emb = embedder.encode(X_train, show_progress_bar=True)
X_test_emb = embedder.encode(X_test, show_progress_bar=True)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1,
)

clf.fit(X_train_emb, y_train)
y_pred = clf.predict(X_test_emb)

print(classification_report(y_test, y_pred, digits=3))


#1.2 Fine-tuning JobBERT on our Seniority Data

In the following we fine-tune JobBERT on our training data - the evaluation is conducted on our test data

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os, json, joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [ ]:
# Load & Clean
SENIORITY_DATA_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/FinalWork/seniority.csv"
df = pd.read_csv(SENIORITY_DATA_PATH)

df = df.copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

print(df.shape)
print(df["label"].value_counts())
df.head()


In [ ]:
# Label Encoding
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"]).astype("int64")

num_labels = len(le.classes_)
label2id = {label: int(i) for i, label in enumerate(le.classes_)}
id2label = {int(i): label for i, label in enumerate(le.classes_)}

# Safety checks
assert not df["label_id"].isna().any(), "label_id enthält NaN"
min_id, max_id = int(df["label_id"].min()), int(df["label_id"].max())
assert min_id >= 0 and max_id < num_labels, f"Label IDs out of range: min={min_id}, max={max_id}, num_labels={num_labels}"

print("num_labels:", num_labels)
print("classes:", le.classes_)


In [ ]:
# Train/Test Split
sen_train, sen_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

sen_train = sen_train.reset_index(drop=True)
sen_test  = sen_test.reset_index(drop=True)

print("train:", sen_train.shape, "test:", sen_test.shape)
print("train label dist:\n", sen_train["label"].value_counts())
print("test label dist:\n", sen_test["label"].value_counts())


In [ ]:
# Datasets
train_ds = Dataset.from_pandas(sen_train[["text", "label_id"]], preserve_index=False)
test_ds  = Dataset.from_pandas(sen_test[["text", "label_id"]], preserve_index=False)

train_ds, test_ds


In [ ]:
# Tokenizer + Tokenizing
MODEL_NAME = "TechWolf/JobBERT-v3"
MAX_LEN = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"]).rename_column("label_id", "labels")
test_ds_tok  = test_ds.map(tokenize, batched=True, remove_columns=["text"]).rename_column("label_id", "labels")

# Torch format (labels als Long)
train_ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer)

train_ds_tok[0]


In [ ]:
# Metrics
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "balanced_accuracy": balanced_accuracy_score(p.label_ids, preds),
        "macro_f1": f1_score(p.label_ids, preds, average="macro"),
    }


In [ ]:
# Model init
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


In [ ]:
# TrainingArguments
training_args = TrainingArguments(
    output_dir="jobbert_seniority_finetuned",
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True
)

In [ ]:
# Trainer + Train + Evaluate
trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_ds_tok,
    eval_dataset=test_ds_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
eval_out = trainer.evaluate()
eval_out


In [ ]:
# Classification Report on Test Split
pred_out = trainer.predict(test_ds_tok)
pred_ids = np.argmax(pred_out.predictions, axis=1)

y_true = le.inverse_transform(pred_out.label_ids)
y_pred = le.inverse_transform(pred_ids)

print("=== FINETUNED JobBERT (Seniority Test) ===")
print(classification_report(y_true, y_pred, digits=3))


In [ ]:
# Save artifacts
SAVE_DIR = "artifacts/jobbert_seniority_finetuned"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

joblib.dump(le, f"{SAVE_DIR}/label_encoder.joblib")
with open(f"{SAVE_DIR}/meta.json", "w") as f:
    json.dump(
        {"threshold": 0.55, "margin": 0.05, "labels": le.classes_.tolist()},
        f, indent=2
    )

print("Saved to:", SAVE_DIR)


#2) Profiles inference + thresholding (OOS)

We apply in the following our fine-tuned model on the profiles dataset

In [ ]:
import pandas as pd

PROFILES_PATH = "https://raw.githubusercontent.com/E-tech-coder/DataScienceCapstoneProject/refs/heads/main/df_profiles_cleansed.csv"
df_profiles = pd.read_csv(PROFILES_PATH)


In [ ]:
# Imports
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import Trainer


In [ ]:
# Tokenize-function for OOS
def tokenize_fn(batch):
    return tokenizer(
        batch["position"],
        truncation=True,
        max_length=MAX_LEN
    )


In [ ]:
# Helper: predict probabilities on any dataframe (Out-of-sample)
def predict_proba_for_positions(trainer: Trainer, df: pd.DataFrame, text_col="position"):
    tmp = df[[text_col]].copy()
    tmp = tmp.rename(columns={text_col: "position"})  # damit tokenize_fn immer "position" findet

    ds = Dataset.from_pandas(tmp, preserve_index=False)
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["position"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    out = trainer.predict(ds)
    logits = out.predictions
    proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return proba


In [ ]:
# Block 3 — Predict on profiles (Seniority)

proba_p = predict_proba_for_positions(trainer, df_profiles, text_col="position")

pred_idx = proba_p.argmax(axis=1)
pred_conf = proba_p[np.arange(len(proba_p)), pred_idx]
pred_label = le.inverse_transform(pred_idx).astype(object)

df_profiles["pred_seniority_raw"] = pred_label
df_profiles["pred_conf"] = pred_conf

df_profiles[["position", "seniority", "pred_seniority_raw", "pred_conf"]].head(10)


#Remark:
As for the department data we have here with "professional" a similar problem - to adress this issue the model rejects the prediction if it falls under a certain threshold and assigns "professional". The assumption here is that if the model wasnt trained on this class the prediction confidence is not high enough and its therefore likely the label is instead professional

In [ ]:
# Threshold + top-2 margin (Seniority)
THRESH = 0.75
USE_MARGIN = True
MARGIN = 0.05

top2 = np.partition(proba_p, -2, axis=1)[:, -2]
margin = pred_conf - top2

pred_final = pred_label.copy()
pred_final[pred_conf < THRESH] = "Professional"

if USE_MARGIN:
    unsure = (pred_final != "Professional") & (margin < MARGIN)
    pred_final[unsure] = "Professional"

df_profiles["pred_seniority"] = pred_final
df_profiles["pred_margin"] = margin

df_profiles[["position", "pred_seniority_raw", "pred_conf", "pred_margin", "pred_seniority"]].head(10)


In [ ]:
def threshold_sweep(y_true, pred_label, pred_conf, thresholds):
    rows = []
    for thr in thresholds:
        pred = pred_label.copy()
        pred[pred_conf < thr] = "Other"
        rows.append({
            "threshold": thr,
            "accuracy": accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "coverage_(not_Other)": float((pred != "Other").mean())
        })
    return pd.DataFrame(rows)

GT_COL = None
for c in ["seniority"]:
    if c in df_profiles.columns:
        GT_COL = c
        break

if GT_COL is not None:
    y_true_p = df_profiles[GT_COL].astype(str).values
    thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

    df_thr = threshold_sweep(
        y_true=y_true_p,
        pred_label=df_profiles["pred_seniority_raw"].astype(object).values,
        pred_conf=df_profiles["pred_conf"].values,
        thresholds=thresholds
    )
    df_thr
else:
    print("No ground-truth seniority column found in df_profiles → threshold sweep not possible.")

In [ ]:
# Classification Report
if GT_COL is not None:
    print("=== PROFILES CLASSIFICATION REPORT (Seniority With Thresholding) ===")
    print(classification_report(
        df_profiles[GT_COL].astype(str).values,
        df_profiles["pred_seniority"].values,
        digits=3,
        zero_division=0
    ))


#2.1 Removing "Professional"
As for department we remove the problem label to observe how our model performs with classes it was trained on

In [ ]:
#Removing Professional
GT_COL = "seniority" if "seniority" in df_profiles.columns else "label"

mask_no_prof = df_profiles[GT_COL] != "Professional"

df_no_prof = df_profiles.loc[mask_no_prof].copy()

print("Remaining samples:", df_no_prof.shape[0])
print(df_no_prof[GT_COL].value_counts())


#2.1.1 Evaluation for fine-tuned model without "Professional"

In [ ]:
from sklearn.metrics import classification_report

print("=== PROFILES CLASSIFICATION REPORT (NO 'Professional') ===")
print(
    classification_report(
        df_no_prof[GT_COL].astype(str).values,
        df_no_prof["pred_seniority"].values,
        digits=3,
        zero_division=0
    )
)


In [ ]:
print("Generating embeddings for df_no_prof...")

X_no_prof = df_no_prof["position"].astype(str).tolist()
X_no_prof_emb = embedder.encode(X_no_prof, show_progress_bar=True)

print("Making predictions with Logistic Regression model...")
y_pred_lr_no_prof = clf.predict(X_no_prof_emb)
y_true_lr_no_prof = df_no_prof[GT_COL].astype(str).values

print("=== PROFILES CLASSIFICATION REPORT (NO 'Professional') — BASELINE (JobBERT embeddings + LogReg) ===")
print(classification_report(y_true_lr_no_prof, y_pred_lr_no_prof, digits=3, zero_division=0))

Conclusion:

In an out-of-sample evaluation on profile data without the "Professional" class, a baseline using frozen JobBERT embeddings with a logistic regression classifier slightly outperforms the fine-tuned model in terms of macro-F1. This behavior is expected under distribution shift, as the baseline learns smoother decision boundaries, while the fine-tuned model is optimized for in-distribution performance and exhibits sharper class boundaries.

We can also observe that the semantic between Senior and Professional seems to be close.

The fine-tuned model, however, enables confidence-based abstention and achieves near-perfect performance in a closed-world setting, highlighting the trade-off between robustness and precision control.